# Direct Pipeline for English-to-Chinese Dialogue Summarization

This notebook implements a local Direct baseline for English-to-Chinese cross-lingual dialogue summarization.

The Direct pipeline uses a single local small language model agent. The agent reads the original English dialogue and directly generates a concise Chinese summary without using an intermediate English summary, full-dialogue translation, semantic representation, or revision step.

```text
English Dialogue
→ Direct Summarization Agent
→ Final Chinese Summary
```

The pipeline consists of one agent:

```text
Direct Summarization Agent
Input: original English dialogue
Output: concise Chinese summary
```
This setup is used as the simplest cross-lingual summarization baseline. Unlike Translate-then-Summarize, Summarize-then-Translate, or the Semantic-agent pipeline, the Direct pipeline performs the task in a single model call.

The local small language model is served through Ollama. The notebook controls the prompt design, input/output processing, direct summary generation, output inspection, and result saving.

## 0. Local Ollama Setup

Before running this notebook, install Ollama and download the local model used for the Direct baseline.

### Recommended model setup

This notebook uses one local small language model through Ollama.

```bash
ollama pull qwen3.5:27b
```

If qwen3.5:27b is too slow on your machine, you can use a smaller model for testing:

```bash
ollama pull "qwen3.5:9b"
```

```bash
DIRECT_MODEL = "qwen3.5:27b"
```

You can check downloaded models with:

```bash
ollama list
```

You can check currently loaded models with:

```bash
ollama ps
```

If the Ollama server is not running, start it with:

```bash
ollama serve
```

On macOS, opening the Ollama app usually starts the local server automatically.


In [1]:
# Cell 1: Install required Python packages.
# Run this only once if the packages are not installed

!pip install requests pandas tqdm


In [1]:
# Cell 2: Imports and global configuration

import json
import time
from pathlib import Path
from typing import Any, Dict, List, Optional

import requests
import pandas as pd
from tqdm.auto import tqdm

OLLAMA_HOST = "http://localhost:11434"

# Direct model
DIRECT_MODEL = "qwen3.5:27b"  # Change this if your local Ollama model name is different

DEFAULT_TEMPERATURE = 0.2
DEFAULT_NUM_CTX = 8192

# Gold set path
GOLD_SET_PATH = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/gold_results/gold_set_50_zh_XSAMSum_bart.json"
)

# Output directory
OUTPUT_DIR = Path(
    "/Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Output files for Direct baseline
FULL_OUTPUT_PATH = OUTPUT_DIR / "direct_qwen27b_50samples.jsonl"
FINAL_CSV_PATH = OUTPUT_DIR / "direct_qwen27b_50samples.csv"
ERROR_OUTPUT_PATH = OUTPUT_DIR / "direct_qwen27b_50samples_errors.jsonl"

print("Gold set path:", GOLD_SET_PATH)
print("Output directory:", OUTPUT_DIR)
print("Full JSONL output path:", FULL_OUTPUT_PATH)
print("Final CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Gold set path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/data/gold_results/gold_set_50_zh_XSAMSum_bart.json
Output directory: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results
Full JSONL output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/direct_qwen27b_50samples.jsonl
Final CSV output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/direct_qwen27b_50samples.csv
Error output path: /Users/yunu919/Desktop/CLMS/LING573/573ChineseEnglishSummarization/notebooks/agents/results/direct_qwen27b_50samples_errors.jsonl


In [3]:
print(GOLD_SET_PATH.exists())

True


In [4]:
# Cell 3: Check whether Ollama is running

def check_ollama_server() -> bool:
    try:
        response = requests.get(f"{OLLAMA_HOST}/api/tags", timeout=10)
        response.raise_for_status()
        models = response.json().get("models", [])

        print("Ollama server is running.")
        print(f"Downloaded models: {[m.get('name') for m in models]}")

        return True

    except Exception as e:
        print("Could not connect to Ollama.")
        print("Make sure Ollama is installed and running.")
        print("Try running this in Terminal:")
        print("  ollama serve")
        print()
        print("Error:", repr(e))

        return False


_ = check_ollama_server()

Ollama server is running.
Downloaded models: ['gemma3:27b', 'qwen3.5:27b']


In [5]:
# Cell 4: Ollama API helper

def call_ollama(
    model: str,
    prompt: str,
    system: Optional[str] = None,
    temperature: float = DEFAULT_TEMPERATURE,
    num_ctx: int = DEFAULT_NUM_CTX,
    timeout: int = 900,
) -> str:
    """Call Ollama's local chat API and return the assistant content."""

    messages = []

    if system:
        messages.append({"role": "system", "content": system})

    messages.append({"role": "user", "content": prompt})

    payload = {
        "model": model,
        "messages": messages,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_ctx": num_ctx,
            "num_predict": 1024,
        },
        "think": False,
    }

    response = requests.post(
        f"{OLLAMA_HOST}/api/chat",
        json=payload,
        timeout=timeout,
    )
    response.raise_for_status()

    data = response.json()
    content = data.get("message", {}).get("content", "")

    if content is None:
        content = ""

    return content.strip()

## 1. Prompt Templates

Each agent is defined as:

```text
Agent = model + role-specific prompt + input/output format
```

In the first version, we use a fixed workflow rather than a fully autonomous agent system.


In [6]:
# Cell 5: Direct prompt template
DIRECT_PROMPT = """You are a cross-lingual dialogue summarization agent.

Your task is to read the following English dialogue and directly generate a concise Chinese summary.

Requirements:
- Generate the summary directly in Chinese.
- Do not first write an English summary.
- Do not translate the full dialogue sentence by sentence.
- Summarize only the main information and final outcome.
- Keep the summary concise and faithful to the dialogue.
- Do not add information that is not stated or clearly implied.
- Do not explain your reasoning.
- Output only the final Chinese summary.

STRICT TRANSLATION RULE:
- You MUST translate ALL English proper nouns and speaker names into standard Chinese characters.
- ABSOLUTELY NO English letters or names should appear in the final Chinese summary.

Conciseness: 
- For simple dialogues, prefer 20-50 Chinese characters. 
- For complex dialogues, allow up to 80 Chinese characters.

English dialogue:
{dialogue}

Chinese summary:
"""

In [7]:
# Cell 6: JSONL utility functions

def append_jsonl(record: Dict[str, Any], path: Path) -> None:
    """Append one record to a JSONL file."""
    path.parent.mkdir(parents=True, exist_ok=True)

    with path.open("a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path: Path) -> List[Dict[str, Any]]:
    """Load a JSONL file into a list of dictionaries."""
    if not path.exists():
        return []

    records = []

    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def load_processed_ids(path: Path) -> set:
    """Return IDs that have already been processed."""
    records = load_jsonl(path)

    return {str(record["id"]) for record in records if "id" in record}

In [8]:
# Cell 7: Direct agent function

def fill_prompt(template: str, replacements: Dict[str, str]) -> str:
    """Replace named placeholders in the prompt."""
    prompt = template

    for key, value in replacements.items():
        prompt = prompt.replace("{" + key + "}", value)

    return prompt


def direct_agent(dialogue: str) -> str:
    """Direct baseline: English dialogue -> Chinese summary."""
    prompt = fill_prompt(
        DIRECT_PROMPT,
        {
            "dialogue": dialogue,
        },
    )

    response = call_ollama(
        model=DIRECT_MODEL,
        prompt=prompt,
        temperature=0.2,
    )

    return response.strip()

## 2. Agent Functions

Each function corresponds to one agent in the pipeline.


In [9]:
# Cell 8: Direct pipeline

def run_direct_pipeline(example: Dict[str, Any]) -> Dict[str, Any]:
    """Run the Direct pipeline: English dialogue -> Chinese summary."""

    sample_id = str(example.get("id", "unknown"))
    dialogue = example["dialogue"]

    reference_english_summary = example.get("reference_english_summary", "")
    reference_chinese_summary = example.get("reference_chinese_summary", "")

    final_chinese_summary = direct_agent(dialogue)

    return {
        "id": sample_id,
        "test_index": example.get("test_index", ""),
        "dialogue": dialogue,
        "reference_english_summary": reference_english_summary,
        "reference_chinese_summary": reference_chinese_summary,
        "final_summary": final_chinese_summary,
        "pipeline": "direct",
        "model": DIRECT_MODEL,
        "num_model_calls": 1,
    }

## 3. Test with Examples

Start with five examples before running the full dataset.  
This is the best way to inspect the intermediate outputs between agents.


In [10]:
# Cell 9: Load first 5 examples from the gold set

def load_examples_from_gold_set(path: Path, n: int = 5) -> List[Dict[str, Any]]:
    """Load the first n examples from the gold-set JSON file."""
    if not path.exists():
        raise FileNotFoundError(f"Gold set not found: {path}")

    with path.open("r", encoding="utf-8") as f:
        raw_data = json.load(f)

    examples = []

    for i, item in enumerate(raw_data[:n]):
        examples.append({
            "id": f"gold_{i+1:05d}",
            "test_index": item.get("test_index", ""),
            "dialogue": item["dialogue"],
            "reference_english_summary": item.get("summary", ""),
            "reference_chinese_summary": item.get("summary_zh", ""),
        })

    return examples


test_data = load_examples_from_gold_set(GOLD_SET_PATH, n=50)

print(f"Loaded {len(test_data)} examples.")
print("First example:")
print(test_data[0])

Loaded 50 examples.
First example:
{'id': 'gold_00001', 'test_index': 23, 'dialogue': "Anne: You were right, he was lying to me :/\nIrene: Oh no, what happened?\nJane: who? that Mark guy?\nAnne: yeah, he told me he's 30, today I saw his passport - he's 40\nIrene: You sure it's so important?\nAnne: he lied to me Irene", 'reference_english_summary': 'Mark lied to Anne about his age. Mark is 40.', 'reference_chinese_summary': '马克向安妮隐瞒了自己的年龄。他40岁了。'}


In [11]:
print(DIRECT_PROMPT)

You are a cross-lingual dialogue summarization agent.

Your task is to read the following English dialogue and directly generate a concise Chinese summary.

Requirements:
- Generate the summary directly in Chinese.
- Do not first write an English summary.
- Do not translate the full dialogue sentence by sentence.
- Summarize only the main information and final outcome.
- Keep the summary concise and faithful to the dialogue.
- Do not add information that is not stated or clearly implied.
- Do not explain your reasoning.
- Output only the final Chinese summary.

STRICT TRANSLATION RULE:
- You MUST translate ALL English proper nouns and speaker names into standard Chinese characters.
- ABSOLUTELY NO English letters or names should appear in the final Chinese summary.

Conciseness: 
- For simple dialogues, prefer 20-50 Chinese characters. 
- For complex dialogues, allow up to 80 Chinese characters.

English dialogue:
{dialogue}

Chinese summary:



In [12]:
# Cell 10: Run the Direct pipeline for the first example

result = run_direct_pipeline(test_data[4])
result

{'id': 'gold_00005',
 'test_index': 66,
 'dialogue': "Joyce: Check this out!\r\nJoyce: <link>\r\nMichael: That's cheap!\r\nEdson: No way! I'm booking my ticket now!! ",
 'reference_english_summary': 'Edson is booking his ticket now.',
 'reference_chinese_summary': '埃德森正在订票。',
 'final_summary': '乔伊斯分享了一个链接，迈克尔认为价格低廉，埃德森随即决定立即预订机票。',
 'pipeline': 'direct',
 'model': 'qwen3.5:27b',
 'num_model_calls': 1}

In [13]:
# Cell 11: Print Direct pipeline result clearly

def print_direct_result(result: Dict[str, Any]) -> None:
    print("=== Original Dialogue ===")
    print(result["dialogue"])
    print()

    print("=== Direct Chinese Summary ===")
    print(result["final_summary"])
    print()

    print("=== Reference English Summary ===")
    print(result["reference_english_summary"])
    print()

    print("=== Reference Chinese Summary ===")
    print(result["reference_chinese_summary"])
    print()

    print("=== Metadata ===")
    print("Pipeline:", result["pipeline"])
    print("Model:", result["model"])
    print("Model calls:", result["num_model_calls"])


print_direct_result(result)

=== Original Dialogue ===
Joyce: Check this out!
Joyce: <link>
Michael: That's cheap!
Edson: No way! I'm booking my ticket now!! 

=== Direct Chinese Summary ===
乔伊斯分享了一个链接，迈克尔认为价格低廉，埃德森随即决定立即预订机票。

=== Reference English Summary ===
Edson is booking his ticket now.

=== Reference Chinese Summary ===
埃德森正在订票。

=== Metadata ===
Pipeline: direct
Model: qwen3.5:27b
Model calls: 1


## 4. Save Results

This saves all intermediate outputs and the final output.


In [14]:
# Cell 12: Reset previous outputs before batch inference

FULL_OUTPUT_PATH.unlink(missing_ok=True)
FINAL_CSV_PATH.unlink(missing_ok=True)
ERROR_OUTPUT_PATH.unlink(missing_ok=True)

print("Previous output files reset.")
print("JSONL output path:", FULL_OUTPUT_PATH)
print("CSV output path:", FINAL_CSV_PATH)
print("Error output path:", ERROR_OUTPUT_PATH)

Previous output files reset.
JSONL output path: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\direct_qwen27b_50samples.jsonl
CSV output path: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\direct_qwen27b_50samples.csv
Error output path: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\direct_qwen27b_50samples_errors.jsonl


## 5. Batch Inference with Checkpointing

This cell processes examples one by one and appends each completed result to `results/agentic_outputs.jsonl`.

If the notebook stops, already processed examples remain saved.


In [ ]:
# Cell 13: Batch inference with Direct pipeline
# Time stamp: 11m 46.1s

MAX_EXAMPLES = 50
SLEEP_SECONDS = 0.2

processed_ids = load_processed_ids(FULL_OUTPUT_PATH)
print(f"Already processed: {len(processed_ids)} examples")

subset = test_data[:MAX_EXAMPLES]

for ex in tqdm(subset, desc="Running Direct pipeline"):
    sample_id = str(ex.get("id", "unknown"))

    if sample_id in processed_ids:
        continue

    try:
        record = run_direct_pipeline(ex)
        append_jsonl(record, FULL_OUTPUT_PATH)
        processed_ids.add(sample_id)
        time.sleep(SLEEP_SECONDS)

    except Exception as e:
        error_record = {
            "id": sample_id,
            "test_index": ex.get("test_index", ""),
            "error": repr(e),
            "dialogue": ex.get("dialogue", ""),
        }

        append_jsonl(error_record, ERROR_OUTPUT_PATH)
        print(f"Error on {sample_id}: {repr(e)}")

print(f"Finished. Outputs saved to: {FULL_OUTPUT_PATH}")

Already processed: 0 examples


Running Direct pipeline:   0%|          | 0/50 [00:00<?, ?it/s]

Finished. Outputs saved to: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\direct_qwen27b_50samples.jsonl


## 6. Export Final Summaries to CSV

This file can be used for ROUGE, BERTScore, OmniScore, or manual analysis.


In [16]:
# Cell 14: Export Direct summaries to CSV

records = load_jsonl(FULL_OUTPUT_PATH)

rows = []

for record in records:
    if "final_summary" not in record:
        continue

    rows.append({
        "id": record.get("id", ""),
        "test_index": record.get("test_index", ""),
        "dialogue": record.get("dialogue", ""),
        "final_summary": record.get("final_summary", ""),
        "reference_english_summary": record.get("reference_english_summary", ""),
        "reference_chinese_summary": record.get("reference_chinese_summary", ""),
        "pipeline": record.get("pipeline", "direct"),
        "model": record.get("model", ""),
        "num_model_calls": record.get("num_model_calls", 1),
    })

df = pd.DataFrame(rows)

if not df.empty:
    df = df.drop_duplicates(subset=["id"], keep="last")

df.to_csv(FINAL_CSV_PATH, index=False, encoding="utf-8-sig")

print(f"Saved final summaries to: {FINAL_CSV_PATH}")
df

Saved final summaries to: C:\Users\FLEXBOY\OneDrive\바탕 화면\yunu\direct_qwen27b_50samples.csv


,id,test_index,dialogue,final_summary,reference_english_summary,reference_chinese_summary,pipeline,model,num_model_calls
0,gold_00001,23,"Anne: You were right, he was lying to me :/\nI...",安妮发现马克在年龄上欺骗了她，他自称三十岁，实则四十岁。尽管艾琳质疑此事的重要性，安妮仍坚持...,Mark lied to Anne about his age. Mark is 40.,马克向安妮隐瞒了自己的年龄。他40岁了。,direct,qwen3.5:27b,1
1,gold_00002,30,"Mary: hey, im kinda broke, lend me a few box\r...",玛丽因缺钱向卡特借钱，卡特表示自己在火车站，一小时后归还。,Mary ran out of money. Carter is going to lend...,玛丽的钱用完了，卡特打算一小时后借给她一点。,direct,qwen3.5:27b,1
2,gold_00003,39,"Tina: I'll tell you something, this Emirate st...",蒂娜称赞阿联酋航空员工形象出众，并抱怨在机场延误一小时后终于赶上回程航班。阿拉随后告知自己正...,Tina will catch the evening flight back home. ...,蒂娜将乘晚间航班回家。阿拉正在去开会的路上。她会让蒂娜知道事情的进展。,direct,qwen3.5:27b,1
3,gold_00004,65,Ana: You sleeping?\r\nCatherine: Not yet.\r\nA...,安娜和凯瑟琳约定明天一起探望奶奶，并互道晚安。,Ana wants to visit grandma tomorrow. Catherine...,安娜明天想去看望奶奶。凯瑟琳会和她一起去。她起床后会给安娜打电话。,direct,qwen3.5:27b,1
4,gold_00005,66,Joyce: Check this out!\r\nJoyce: <link>\r\nMic...,乔伊斯分享了一个链接，迈克尔认为价格很便宜，埃德森随即决定立即预订机票。,Edson is booking his ticket now.,埃德森正在订票。,direct,qwen3.5:27b,1
5,gold_00006,67,Jane: google maps says it is at least 3h <file...,简因担心路况，提议将见面时间提前至 4 点半，史蒂文同意。两人最终确认在正门见面。,Jane wants to leave at 4.30 instead of 5 becau...,简想4点半就走，而不是等到5点，因为谷歌地图提示300公里的车程至少需要3小时，她不想迟到。...,direct,qwen3.5:27b,1
6,gold_00007,78,"Fiona: Are you free?\r\nTina: Yes, what's up?\...",菲奥娜想为克里斯准备蒂娜的塔塔，但馅料做成了炒蛋。蒂娜答应帮忙，并解释需持续搅拌才能避免失败。,Fiona wants to prepare dinner for Chris. She i...,菲奥娜想为克里斯准备晚餐。她想起了蒂娜做的馅饼。蒂娜会帮她做的。,direct,qwen3.5:27b,1
7,gold_00008,86,Olafur: are we doing anything for New Year's E...,奥拉夫、娜塔莉和佐伊决定新年夜去苏荷区参加蒂凡尼早餐派对，并需尽快购票。,"Nathalie, Olafur and Zoe are planning the New ...",娜塔莉、奥拉维尔和佐伊正在做新年前夜的计划。娜塔莉想要有格调的。但奥拉维尔不喜欢歌剧。他们想...,direct,qwen3.5:27b,1
8,gold_00009,120,"John: wanna go see ""A Star is Born"" on Wed?\r\...",约翰提议周三看电影，琼因忙碌拒绝，但周四有空。两人最终约定周四晚上八点观影，由约翰负责查询场...,"Joan and John are going to watch ""A Star is Bo...",琼和约翰星期四晚上8点左右去看《一个明星的诞生》。,direct,qwen3.5:27b,1
9,gold_00010,137,Peyton: I have been asking you to bring that v...,佩顿催促卡梅伦带回电子游戏，卡梅伦因需在外地多待一周无法回家。佩顿建议通过快递寄送，卡梅伦未...,Peyton is expecting Cameron to bring the video...,佩顿希望卡梅隆能带游戏机过来，但是卡梅隆可能还要再缺席一周。,direct,qwen3.5:27b,1


In [17]:
# Cell 15: Compare generated Chinese summary with the reference Chinese summary

comparison_columns = [
    "id",
    "final_summary",
    "reference_chinese_summary",
]

comparison_df = df[comparison_columns].copy()

comparison_df

,id,final_summary,reference_chinese_summary
0,gold_00001,安妮发现马克在年龄上欺骗了她，他自称三十岁，实则四十岁。尽管艾琳质疑此事的重要性，安妮仍坚持...,马克向安妮隐瞒了自己的年龄。他40岁了。
1,gold_00002,玛丽因缺钱向卡特借钱，卡特表示自己在火车站，一小时后归还。,玛丽的钱用完了，卡特打算一小时后借给她一点。
2,gold_00003,蒂娜称赞阿联酋航空员工形象出众，并抱怨在机场延误一小时后终于赶上回程航班。阿拉随后告知自己正...,蒂娜将乘晚间航班回家。阿拉正在去开会的路上。她会让蒂娜知道事情的进展。
3,gold_00004,安娜和凯瑟琳约定明天一起探望奶奶，并互道晚安。,安娜明天想去看望奶奶。凯瑟琳会和她一起去。她起床后会给安娜打电话。
4,gold_00005,乔伊斯分享了一个链接，迈克尔认为价格很便宜，埃德森随即决定立即预订机票。,埃德森正在订票。
5,gold_00006,简因担心路况，提议将见面时间提前至 4 点半，史蒂文同意。两人最终确认在正门见面。,简想4点半就走，而不是等到5点，因为谷歌地图提示300公里的车程至少需要3小时，她不想迟到。...
6,gold_00007,菲奥娜想为克里斯准备蒂娜的塔塔，但馅料做成了炒蛋。蒂娜答应帮忙，并解释需持续搅拌才能避免失败。,菲奥娜想为克里斯准备晚餐。她想起了蒂娜做的馅饼。蒂娜会帮她做的。
7,gold_00008,奥拉夫、娜塔莉和佐伊决定新年夜去苏荷区参加蒂凡尼早餐派对，并需尽快购票。,娜塔莉、奥拉维尔和佐伊正在做新年前夜的计划。娜塔莉想要有格调的。但奥拉维尔不喜欢歌剧。他们想...
8,gold_00009,约翰提议周三看电影，琼因忙碌拒绝，但周四有空。两人最终约定周四晚上八点观影，由约翰负责查询场...,琼和约翰星期四晚上8点左右去看《一个明星的诞生》。
9,gold_00010,佩顿催促卡梅伦带回电子游戏，卡梅伦因需在外地多待一周无法回家。佩顿建议通过快递寄送，卡梅伦未...,佩顿希望卡梅隆能带游戏机过来，但是卡梅隆可能还要再缺席一周。


In [18]:
# Cell 16: Inspect Direct outputs

if not df.empty:
    inspection_columns = [
        "id",
        "test_index",
        "dialogue",
        "final_summary",
        "reference_chinese_summary",
    ]

    inspection_df = df[inspection_columns].copy()
    display(inspection_df)
else:
    print("No results found.")

,id,test_index,dialogue,final_summary,reference_chinese_summary
0,gold_00001,23,"Anne: You were right, he was lying to me :/\nI...",安妮发现马克在年龄上欺骗了她，他自称三十岁，实则四十岁。尽管艾琳质疑此事的重要性，安妮仍坚持...,马克向安妮隐瞒了自己的年龄。他40岁了。
1,gold_00002,30,"Mary: hey, im kinda broke, lend me a few box\r...",玛丽因缺钱向卡特借钱，卡特表示自己在火车站，一小时后归还。,玛丽的钱用完了，卡特打算一小时后借给她一点。
2,gold_00003,39,"Tina: I'll tell you something, this Emirate st...",蒂娜称赞阿联酋航空员工形象出众，并抱怨在机场延误一小时后终于赶上回程航班。阿拉随后告知自己正...,蒂娜将乘晚间航班回家。阿拉正在去开会的路上。她会让蒂娜知道事情的进展。
3,gold_00004,65,Ana: You sleeping?\r\nCatherine: Not yet.\r\nA...,安娜和凯瑟琳约定明天一起探望奶奶，并互道晚安。,安娜明天想去看望奶奶。凯瑟琳会和她一起去。她起床后会给安娜打电话。
4,gold_00005,66,Joyce: Check this out!\r\nJoyce: <link>\r\nMic...,乔伊斯分享了一个链接，迈克尔认为价格很便宜，埃德森随即决定立即预订机票。,埃德森正在订票。
5,gold_00006,67,Jane: google maps says it is at least 3h <file...,简因担心路况，提议将见面时间提前至 4 点半，史蒂文同意。两人最终确认在正门见面。,简想4点半就走，而不是等到5点，因为谷歌地图提示300公里的车程至少需要3小时，她不想迟到。...
6,gold_00007,78,"Fiona: Are you free?\r\nTina: Yes, what's up?\...",菲奥娜想为克里斯准备蒂娜的塔塔，但馅料做成了炒蛋。蒂娜答应帮忙，并解释需持续搅拌才能避免失败。,菲奥娜想为克里斯准备晚餐。她想起了蒂娜做的馅饼。蒂娜会帮她做的。
7,gold_00008,86,Olafur: are we doing anything for New Year's E...,奥拉夫、娜塔莉和佐伊决定新年夜去苏荷区参加蒂凡尼早餐派对，并需尽快购票。,娜塔莉、奥拉维尔和佐伊正在做新年前夜的计划。娜塔莉想要有格调的。但奥拉维尔不喜欢歌剧。他们想...
8,gold_00009,120,"John: wanna go see ""A Star is Born"" on Wed?\r\...",约翰提议周三看电影，琼因忙碌拒绝，但周四有空。两人最终约定周四晚上八点观影，由约翰负责查询场...,琼和约翰星期四晚上8点左右去看《一个明星的诞生》。
9,gold_00010,137,Peyton: I have been asking you to bring that v...,佩顿催促卡梅伦带回电子游戏，卡梅伦因需在外地多待一周无法回家。佩顿建议通过快递寄送，卡梅伦未...,佩顿希望卡梅隆能带游戏机过来，但是卡梅隆可能还要再缺席一周。
